# 1) Import Libraries

In [1]:
import random
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from collections import Counter
from textblob import TextBlob
from sklearn.metrics import classification_report, accuracy_score

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/kentbaluyot/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/kentbaluyot/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# 2) Creating the Dataset: Hotel & Travel Experience Feedback


## Background & Context
This dataset consists of 100 synthetically generated hotel and travel reviews.

The background of this data simulates a scenario where a hotel management team needs to process high volumes of guest feedback. Instead of reading every full review, the objective is to use a model to:
* **Summarize:** Extract the most "meaningful" sentence that captures the essence of the guest's stay.
* **Analyze:** Determine the sentiment of that specific top sentence to identify service gaps or highlights.

## Dataset Objective
The text is specifically engineered to contain **3 to 5 sentences per row**. This ensures that the **Sentence Scoring** logic has enough comparative data to determine a "top sentence," rather than simply processing a single-sentence entry.

## Column Schema

| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| **email_num** | Integer | A unique identifier for each review entry, starting from 1 to 100. |
| **paragraph** | String | A multi-sentence paragraph containing 3 to 5 sentences describing a guest's experience at a hotel or attraction. |

## Linguistic Properties
* **Tone:** Varies between highly positive, highly negative, and neutral/mixed.
* **Structure:** Each sentence is terminated by a period (`.`) to facilitate clean sentence tokenization.
* **Stop Word Density:** Contains a natural distribution of English stop words (is, the, was) to test the effectiveness of the filtering stage.

In [2]:
# Positive Reviews (34)
positive_reviews = [
    "The check-in process was incredibly smooth and welcoming. The front desk staff offered us a glass of champagne upon arrival, which set a luxury tone for the entire stay. Our luggage was delivered to the room before we even finished our drinks.",
    "Our room featured a breathtaking view of the city skyline that looked even better at night. The floor-to-ceiling windows were spotless, allowing for perfect photography opportunities. We spent hours just watching the city lights from the comfort of our king-sized bed.",
    "The housekeeping team deserves a standing ovation for their attention to detail. Every day, the room was returned to a pristine state with fresh linens and replenished high-end toiletries. We particularly loved the small chocolates left on our pillows during the evening turndown.",
    "Breakfast at the hotel restaurant exceeded all of our expectations. There was a wide variety of fresh fruits, artisanal pastries, and made-to-order omelets. The coffee was high-quality Arabica, and the service was prompt despite the morning rush.",
    "We found the location of the hotel to be absolutely perfect for sightseeing. Most of the major historical landmarks were within a ten-minute walk of the front lobby. This saved us a significant amount of money on ride-sharing and public transportation.",
    "The rooftop pool area felt like a private oasis in the middle of a bustling city. The water temperature was perfectly regulated, and there were plenty of plush lounge chairs available. The poolside bar served some of the best mojitos we have ever tasted.",
    "I was highly impressed by the modern technology integrated into the room design. We could control the lighting, temperature, and curtains all from a central tablet next to the bed. The Wi-Fi signal was also exceptionally strong, which was vital for my remote work.",
    "The hotel gym was better equipped than most commercial fitness centers I've visited. It featured brand-new cardio machines, a full rack of dumbbells, and even a dedicated yoga space. Having chilled eucalyptus towels available after a workout was a very thoughtful touch.",
    "We celebrated our wedding anniversary here, and the staff went out of their way to make it special. They surprised us with a complimentary bottle of wine and a handwritten note in our room. It is rare to find such personalized service in a large hotel chain.",
    "The bathroom was a highlight of the suite, featuring a deep soaking tub and a rainfall showerhead. The water pressure was excellent, and the towels were thick and incredibly soft. It felt like having a private spa experience right in our own room.",
    "Despite being located on a busy street, the soundproofing in the room was top-notch. We didn't hear a single horn honk or siren throughout our entire three-night stay. It provided the quiet, restful environment we desperately needed.",
    "The concierge was an absolute fountain of knowledge regarding local hidden gems. He recommended a small family-owned bistro that wasn't on any of the popular \"must-visit\" lists. It ended up being the best meal of our entire trip.",
    "We were traveling with our golden retriever, and the hotel was incredibly pet-friendly. They provided a comfortable dog bed, bowls, and even a small bag of organic treats. It made traveling with a pet feel effortless and welcoming.",
    "The interior design of the lobby was sophisticated and visually stunning. The combination of marble floors and contemporary art created an atmosphere of refined elegance. It was a beautiful place to sit and enjoy a cocktail before heading out.",
    "Every staff member we encountered greeted us with a genuine smile and a \"hello.\" It didn't feel forced or scripted, but rather like they truly enjoyed their jobs. This level of hospitality is what will definitely bring us back next year.",
    "The valet parking service was fast and efficient every time we needed our car. Even during the checkout peak, we only had to wait about five minutes for our vehicle. The drivers were professional and handled our car with great care.",
    "I appreciated the eco-friendly initiatives the hotel has implemented throughout the property. They use glass water bottles instead of plastic and have a very clear recycling program in the rooms. It's nice to stay at a place that prioritizes sustainability.",
    "The bed was hands down the most comfortable hotel bed I have ever slept in. The mattress had the perfect level of firmness, and the pillows were incredibly supportive. I woke up every morning feeling refreshed and free of back pain.",
    "We spent a rainy afternoon in the hotel library, which was a cozy and quiet retreat. They had a great selection of books and comfortable leather armchairs. It was the perfect spot to relax with a cup of hot tea.",
    "The evening jazz band in the hotel lounge was a fantastic addition to our stay. The music was at a perfect volume where we could still hold a conversation while enjoying the performance. It added a layer of class and entertainment to our nights.",
    "Security at the hotel made us feel very safe throughout our visit. You needed a key card to access the elevators and the residential floors, which we appreciated. The lobby was also well-staffed at all hours of the night.",
    "The airport shuttle service was punctual and the van was very clean and spacious. The driver was helpful with our heavy suitcases and gave us some great tips on which terminal gate to head toward. It took all the stress out of our departure.",
    "Our room's balcony was the perfect spot for a morning coffee. We had a great view of the courtyard gardens, which were meticulously landscaped and very peaceful. It was a wonderful way to start each day of our vacation.",
    "The pricing for the mini-bar was surprisingly reasonable for a high-end hotel. Usually, these items are marked up excessively, but here they were fair. It was nice to grab a snack without feeling like I was being overcharged.",
    "We were impressed by the speed of the laundry and dry-cleaning service. We dropped off a suit in the morning and it was returned, perfectly pressed, by the same evening. This efficiency is exactly what a business traveler needs.",
    "The spa's salt room was an incredibly relaxing experience that I hadn't tried before. It was the perfect way to detox after a long flight and a busy day of meetings. The staff provided warm herbal tea and soft robes that made the experience even better.",
    "The hotel's breakfast buffet featured a live cooking station where the chef made personalized crepes. I loved being able to choose my own fillings, and the ingredients were all incredibly fresh. It was a delightful gourmet touch to a standard meal.",
    "We were pleasantly surprised by the library of movies available for free on our room's smart TV. It was a great way to unwind in the evening without having to pay for extra entertainment. The sound system in the room was also surprisingly high quality.",
    "The walkability of the neighborhood around the hotel was a major plus for us. We discovered several charming bookstores and cafes within just a few blocks of the entrance. It really made us feel like we were living like locals during our stay.",
    "The hotel's commitment to supporting local artists was evident in the beautiful murals throughout the common areas. Each piece had a small plaque explaining the artist's inspiration and background. It added a unique and meaningful touch to the hotel's aesthetic.",
    "We loved the fact that the hotel provided high-quality sunscreen and after-sun lotion at the pool area. It's those small, thoughtful gestures that really show they care about their guests' well-being. We didn't have to worry about a thing while lounging in the sun.",
    "The hotel's executive lounge offered a stunning view of the harbor that was especially beautiful at sunset. The evening hors d'oeuvres were sophisticated and varied, providing a great pre-dinner snack. The staff in the lounge were attentive and professional.",
    "Our room was equipped with a high-end air purification system that made the air feel incredibly fresh. This was a huge plus for my allergies and helped me sleep much better than I usually do in hotels. It's a great feature for health-conscious travelers.",
    "The hotel's staff were exceptionally helpful when we needed to arrange a complicated series of tours. They handled all the bookings and transportation, making our itinerary seamless and stress-free. We couldn't have asked for better assistance.",
]

# Neutral Reviews (33)
neutral_reviews = [
    "The room was generally clean and the bed was comfortable enough for a two-night stay. However, the decor felt a bit dated and could use a modern refresh. It served its purpose but didn't necessarily wow us.",
    "Check-in was straightforward, though the staff seemed a bit preoccupied with other tasks. The location is decent, but there aren't many dining options within immediate walking distance. It's a fair choice for the price point.",
    "The Wi-Fi worked for basic browsing, but I struggled with video calls during peak evening hours. The gym was small but functional, containing the basics like a treadmill and some free weights. It's an average experience overall.",
    "Our stay was fine, but nothing particularly stood out as exceptional. The breakfast buffet had a standard selection of cereal and eggs, but the quality was just middle-of-the-road. We might stay here again if the price is right.",
    "The hotel is located in a quiet neighborhood, which made for good sleep. On the downside, it's quite a trek to the city center using public transport. It's a trade-off between peace and convenience.",
    "Housekeeping came every day, but they often forgot to replace the used coffee pods. The room size was adequate for two people, though the closet was a bit cramped. It was a functional, if uninspired, stay.",
    "The lobby area is quite grand, but the actual guest rooms feel much more basic. I found the air conditioning to be a bit loud when it cycled on and off at night. It's a decent three-star property.",
    "We used the pool once, and it was fine, though the water was a little colder than I prefer. There were enough towels, but the area could have used a quick sweep. It's a standard amenity for this level of hotel.",
    "The staff were polite when spoken to but didn't go out of their way to offer assistance. The room was quiet, which I appreciated, but the view was just of a brick wall. It's an okay place for a short business trip.",
    "The breakfast was included in our rate, which was nice, but the hot food was often lukewarm. The dining room was also quite crowded during the peak morning hour. It was a passable meal to start the day.",
    "I liked that they offered a free shuttle to the airport, even if it only ran once an hour. The room was okay, but the carpet looked like it had seen better days. It's an affordable option for a quick layover.",
    "The bathroom was clean, though the lighting was a bit dim for shaving. The bed was firm, which some might like, but I found it a little stiff. It's a standard budget-friendly hotel experience.",
    "We found the elevators to be quite slow during the morning rush. The room itself was tidy, but we could hear occasional muffled talking from the neighboring room. It's a middle-tier hotel that meets basic needs.",
    "Parking was available on-site for a daily fee, which was convenient but a bit expensive. The staff at the front desk were efficient but lacked a personal touch. It's a professional, business-like environment.",
    "The in-room coffee was okay, though I ended up going to a local cafe instead. The pillows were a bit too soft for my liking, but the linens were clean. It was a perfectly average stay without any major issues.",
    "The hotel is located right next to a shopping mall, which is convenient for errands. However, the area gets very congested with traffic during the day. It's a functional location for specific needs.",
    "Our room was ready on time, but we weren't offered any information about the hotel's amenities at check-in. The gym was empty when I went, but half of the machines were out of order. It was an okay stay for the price.",
    "The water pressure in the shower was a bit weak, but the temperature was consistent. The TV had a good selection of channels, which was nice for a quiet night in. It's a standard hotel that doesn't take many risks.",
    "The furniture in the room was functional but showed some noticeable wear and tear on the corners. The staff handled our checkout quickly, which was helpful for our early flight. It's a reliable, if plain, choice.",
    "The hotel bar had a limited selection of drinks but the service was fast. The lobby seating was comfortable enough while waiting for our taxi. It's a typical urban hotel with no major surprises.",
    "I found the room temperature hard to regulate; it was either a bit too cold or too warm. The towels were a bit scratchy, but they were replaced daily. It's a fine place to stay if you just need a bed for the night.",
    "The location is excellent for the convention center, but the surrounding area is a bit deserted at night. The room was clean and the staff were helpful with directions. It's a practical choice for business travelers.",
    "Breakfast was a mix of hits and misses; the pastries were fresh but the eggs were quite bland. The seating area was clean, though the background music was a bit repetitive. It's an average morning experience.",
    "The lobby smells very nice, but that scent doesn't quite carry over into the hallways. The room was a bit small for the price, but the location made up for it. It's a fair deal overall.",
    "We requested extra blankets and they were delivered within twenty minutes. The room design was a bit clunky, with the light switches in odd places. It was a standard stay that checked the basic boxes.",
    "The Wi-Fi was free, but the speed was only sufficient for checking emails. The bathroom was small but had a decent amount of counter space. It's an unremarkable but perfectly safe option.",
    "I appreciated the tea-making facilities in the room, though the selection of teas was limited. The bed was made every day, but the duvet felt a bit thin. It's a standard accommodation for a traveler on a budget.",
    "The sound from the hallway was occasionally audible, but it wasn't loud enough to keep us awake. The room was bright with natural light, which was a plus. It's a decent, middle-of-the-road hotel.",
    "Check-out was at 11:00 AM, which felt a little early compared to other places I've stayed. The room was clean and the staff were professional during the brief interaction. It's a standard corporate-style hotel.",
    "The fitness center is only open during limited hours, which was a bit inconvenient for my schedule. The room was basic but functional, with a small desk and chair. It's a fine choice for a short-term stay.",
    "The lobby décor is a bit flashy for my taste, but it was well-maintained. The staff were efficient and got us to our room quickly. It's a solid, predictable hotel for a one-night stopover.",
    "I found the hotel's restaurant to be a bit overpriced for the quality of food served. The atmosphere was nice, but the menu was fairly limited. It's an okay option if you don't want to leave the property.",
    "The room's décor was a bit corporate and lacked any local character. However, it was clean and the bed provided a decent night's sleep. It's a standard business hotel that does what it's supposed to do.",
]

# Negative Reviews (33)
negative_reviews = [
    "I was extremely disappointed to find that the room I booked was not the one I received. The staff were unhelpful when I pointed out the discrepancy and refused to move me. The air conditioning was broken and the room was stiflingly hot all night.",
    "The cleanliness of the room was unacceptable; I found hair in the shower and stains on the bed sheets. When I called the front desk, they told me no housekeeping was available until the morning. I would never recommend this place to anyone.",
    "Our stay was ruined by the constant noise from a construction project right outside our window that started at 6:00 AM. The hotel failed to mention this during the booking process or at check-in. It was impossible to get any rest.",
    "The Wi-Fi was advertised as high-speed but was virtually non-existent in our room. I had to go down to the lobby just to send a simple email, which was incredibly frustrating. For the price they charge, this is simply not acceptable.",
    "I had a terrible experience with the valet service; they lost my car keys for over an hour. Instead of apologizing, the staff were rude and acted like it was my fault. I almost missed my flight because of their incompetence.",
    "The breakfast was a disaster, with most of the food trays being empty even an hour before the service ended. The tables were sticky and hadn't been cleared from previous guests. It was a very unhygienic and poorly managed experience.",
    "The \"mountain view\" I paid extra for was actually just a view of the hotel's trash bins and a parking lot. When I asked for a refund of the price difference, the manager was incredibly dismissive. I feel completely cheated by their false advertising.",
    "The elevator broke down while we were inside, and it took over thirty minutes for anyone to respond to the alarm. The staff seemed completely untrained for emergencies and offered no compensation for the ordeal. It was a terrifying experience.",
    "There was a strong smell of cigarette smoke in our \"non-smoking\" room that made it very difficult to sleep. Despite our complaints, the hotel was full and they couldn't move us to another room. We left with a headache the next morning.",
    "The plumbing in the bathroom was ancient; the toilet clogged twice and the sink drained at a snail's pace. It took three calls to maintenance before anyone finally showed up to fix the issue. The whole experience was incredibly inconvenient.",
    "I was shocked to find hidden fees on my bill at checkout that were never mentioned during booking. The staff were very aggressive when I questioned the charges and refused to explain them. I will definitely be disputing this with my credit card company.",
    "The pool area was filthy, with trash overflowing from the bins and used towels scattered everywhere. It looked like it hadn't been cleaned in days, and the water was cloudy and uninviting. It was a complete waste of an amenity.",
    "The walls were so thin that I could hear every word of the conversation in the next room. It felt like I had zero privacy and the noise kept me up until the early hours of the morning. This is by far the worst night's sleep I've ever had.",
    "The front desk staff were incredibly rude and acted like my presence was a major inconvenience to them. I stood at the counter for ten minutes before anyone even acknowledged me. I've never experienced such poor customer service in my life.",
    "Our room was never cleaned during our entire three-night stay, despite us leaving the \"please clean\" sign on the door. When I mentioned it to the manager, they simply shrugged and said they were short-staffed. It was a totally unprofessional experience.",
    "The hotel advertised a shuttle service to the beach, but when we tried to use it, we were told it had been discontinued months ago. This was a major reason we booked this hotel, and we felt misled. We ended up spending a lot of money on taxis.",
    "I found a cockroach in the bathroom on our second night, which was absolutely disgusting. When I reported it, the staff just gave me a can of bug spray instead of moving us to a new room. I couldn't wait to check out and leave.",
    "The \"king-sized\" bed was actually just two twin mattresses pushed together, with a large gap in the middle. It was incredibly uncomfortable and led to a very poor night's sleep. This is not what I expected from a hotel at this price point.",
    "The gym was a joke, with only one working treadmill and a set of rusted dumbbells. The room was also poorly ventilated and smelled of mildew. It was definitely not the \"state-of-the-art\" facility they promised on their website.",
    "Our room was located right next to the service elevator, and the noise from staff throughout the night was constant. We were never told about this potential issue when we were assigned the room. It was a very noisy and unpleasant stay.",
    "The hotel's restaurant service was incredibly slow; it took over an hour for our appetizers to arrive. When the main courses finally came, they were cold and poorly cooked. We ended up leaving without finishing our meal.",
    "I was charged for several items from the minibar that I never even touched. When I tried to resolve the issue at checkout, the clerk was very unhelpful and insisted I must have used them. It was a very frustrating and dishonest experience.",
    "The bathroom tiles were cracked and there was mold growing around the edges of the tub. It was clear that the room hadn't been properly maintained in quite some time. The overall lack of upkeep was very disappointing.",
    "The hotel's \"luxury\" spa was a major letdown, with rude staff and a very clinical atmosphere. The massage I received was rushed and not at all relaxing. It was definitely not worth the high price I paid.",
    "We were told our room would be ready by 3:00 PM, but we weren't actually able to check in until after 6:00 PM. No apology or compensation was offered for the long wait. It was a very poor start to our vacation.",
    "The carpet in our room was damp and had a strange, musty odor that was very unpleasant. We tried to open a window to let in some fresh air, but it was stuck shut. It made the entire stay feel very uncomfortable.",
    "The hotel's \"free\" parking was actually a tiny, crowded lot that was nearly impossible to navigate. I ended up scraping my car on a concrete pillar while trying to park. It was a very stressful and poorly managed situation.",
    "I was woken up at 3:00 AM by a loud party in the room above us that went on for hours. When I called security, they said there was nothing they could do about it. It was a completely sleepless and miserable night.",
    "The hotel's \"concierge\" was just a desk with a few old brochures and no actual person to help us. We had to figure everything out on our own, which was very disappointing for a hotel that claims to be high-end.",
    "The air conditioning in the lobby was so cold that it was uncomfortable to sit there for more than a few minutes. Conversely, the air conditioning in our room was barely working at all. The entire climate control system seemed to be failing.",
    "I found the hotel's \"security\" to be very lax, with the side doors left unlocked at all hours of the night. This made me feel very unsafe, especially given the neighborhood. I would not stay here again for safety reasons.",
    "The hotel's \"complimentary\" breakfast was just some stale bread and instant coffee. It was a far cry from the \"gourmet\" spread they showed in the photos online. We ended up going to a nearby diner every morning.",
    "The hotel's \"business center\" consisted of one ancient computer that was incredibly slow and a printer that was out of ink. It was impossible to get any work done, which was a major problem for my trip.",
]

# Create labeled review tuples BEFORE shuffling
labeled_reviews = []
labeled_reviews.extend([(review, 'Positive') for review in positive_reviews])
labeled_reviews.extend([(review, 'Neutral') for review in neutral_reviews])
labeled_reviews.extend([(review, 'Negative') for review in negative_reviews])

# Shuffle the labeled reviews (labels stay with their reviews)
random.shuffle(labeled_reviews)

# Create a list of dictionaries with email_num and paragraph
data = []
for i in range(1, len(labeled_reviews) + 1):
    data.append({
        "email_num": i,
        "paragraph": labeled_reviews[i - 1][0],
        "actual_sentiment": labeled_reviews[i - 1][1]
    })

# Create DataFrame and save to CSV
df = pd.DataFrame(data)
df.to_csv("hotel_reviews.csv", index=False)

print(f"Success! 'hotel_reviews.csv' has been created with {len(df)} rows in random order.")
print(f"Total reviews: {len(positive_reviews)} positive + {len(neutral_reviews)} neutral + {len(negative_reviews)} negative = {len(labeled_reviews)} total")
print(f"\nSentiment distribution in shuffled data:")
print(df['actual_sentiment'].value_counts())

Success! 'hotel_reviews.csv' has been created with 100 rows in random order.
Total reviews: 34 positive + 33 neutral + 33 negative = 100 total

Sentiment distribution in shuffled data:
actual_sentiment
Positive    34
Neutral     33
Negative    33
Name: count, dtype: int64


# 3) Load Dataset

In [3]:
file_path = "hotel_reviews.csv"
df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print()
df.head()

Dataset loaded successfully!
Shape: (100, 3)



,email_num,paragraph,actual_sentiment
0,1,I appreciated the eco-friendly initiatives the...,Positive
1,2,"Our room was ready on time, but we weren't off...",Neutral
2,3,We spent a rainy afternoon in the hotel librar...,Positive
3,4,Our room's balcony was the perfect spot for a ...,Positive
4,5,The water pressure in the shower was a bit wea...,Neutral


# 4) Natural Language Processing

## 4.1 Sentence Tokenization

In [4]:
df['sentences'] = df['paragraph'].apply(sent_tokenize)
# Preview the first 5 entry's sentences
df['sentences'].head()

0    [I appreciated the eco-friendly initiatives th...
1    [Our room was ready on time, but we weren't of...
2    [We spent a rainy afternoon in the hotel libra...
3    [Our room's balcony was the perfect spot for a...
4    [The water pressure in the shower was a bit we...
Name: sentences, dtype: object

## 4.2 Word Tokenization

In [5]:
df['words'] = df['paragraph'].apply(lambda x: word_tokenize(x.lower()))
# Preview the first 5 entry's words
df['words'].head()

0    [i, appreciated, the, eco-friendly, initiative...
1    [our, room, was, ready, on, time, ,, but, we, ...
2    [we, spent, a, rainy, afternoon, in, the, hote...
3    [our, room, 's, balcony, was, the, perfect, sp...
4    [the, water, pressure, in, the, shower, was, a...
Name: words, dtype: object

## 4.3 Stop Word Removal

In [6]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(word_list):
    return [word for word in word_list if word.isalnum() and word not in stop_words]

df['filtered_words'] = df['words'].apply(remove_stopwords)
# Preview filtered words
df['filtered_words'].head()

0    [appreciated, initiatives, hotel, implemented,...
1    [room, ready, time, offered, information, hote...
2    [spent, rainy, afternoon, hotel, library, cozy...
3    [room, balcony, perfect, spot, morning, coffee...
4    [water, pressure, shower, bit, weak, temperatu...
Name: filtered_words, dtype: object

In [7]:
## 4.4 Global Word Frequency Calculation

# Calculate word frequencies across ALL reviews in the corpus
# This gives more stable frequency estimates than per-review frequencies
all_filtered_words = []
for words_list in df['filtered_words']:
    all_filtered_words.extend(words_list)

global_word_freq = Counter(all_filtered_words)

print(f"Total unique words in corpus: {len(global_word_freq)}")
print(f"Most common words (top 15):")
for word, count in global_word_freq.most_common(15):
    print(f"  {word}: {count}")

Total unique words in corpus: 1025
Most common words (top 15):
  room: 48
  hotel: 46
  staff: 21
  stay: 19
  bit: 19
  us: 17
  night: 14
  incredibly: 14
  like: 13
  service: 12
  bed: 12
  experience: 12
  made: 12
  morning: 11
  could: 10


## 4.4.1 Custom Sentiment Lexicon (High-Value Words)

In [ ]:
# High-value sentiment words (3x weight during scoring)
high_value_positive = {
    'exceptional', 'excellent', 'outstanding', 'amazing', 'wonderful', 'fantastic',
    'breathtaking', 'pristine', 'spotless', 'impressive', 'delightful',
    'beautiful', 'gorgeous', 'perfect', 'phenomenal', 'incredible', 'superb'
}

high_value_negative = {
    'disgusting', 'filthy', 'unacceptable', 'terrible', 'horrible', 'awful',
    'broken', 'dirty', 'rude', 'disappointing', 'incompetent', 'disgraceful',
    'unpleasant', 'nightmarish', 'abysmal', 'dreadful', 'unfit', 'unsatisfactory'
}

high_value_sentiment = high_value_positive.union(high_value_negative)

# Broader adjective list for sentiment density (1.5x multiplier)
adjectives = high_value_sentiment.union({
    'good', 'bad', 'great', 'poor', 'clean', 'uncomfortable', 'friendly',
    'unhelpful', 'smooth', 'rough', 'nice', 'fresh', 'stale', 'spacious',
    'cramped', 'quiet', 'noisy', 'warm', 'cold', 'professional', 'efficient',
    'slow', 'fast', 'sophisticated', 'modern', 'dated', 'thoughtful', 'lukewarm'
})

print(f"Sentiment Lexicon Loaded:")
print(f"  High-value positive words: {len(high_value_positive)}")
print(f"  High-value negative words: {len(high_value_negative)}")
print(f"  Total adjectives: {len(adjectives)}")

High-value sentiment lexicon loaded:
  Positive words: 17
  Negative words: 18
  Total weight-3x words: 35
  Total adjectives for density scoring: 63


## 4.4.2 Word Frequency Calculation

In [9]:
# Display global word frequency statistics
print(f"\nGlobal word frequency calculated from entire corpus")
print(f"Sample of word frequencies:")
print(global_word_freq.most_common(10))


Global word frequency calculated from entire corpus
Sample of word frequencies:
[('room', 48), ('hotel', 46), ('staff', 21), ('stay', 19), ('bit', 19), ('us', 17), ('night', 14), ('incredibly', 14), ('like', 13), ('service', 12)]


## 4.5 Sentence Scoring

In [ ]:
def detect_negation(sentence, word_index):
    """Check if a word is preceded by negation (not, no, never)."""
    words = word_tokenize(sentence.lower())
    negation_words = {'not', 'no', 'never', 'neither', 'nowhere', 'nobody'}
    
    for i in range(max(0, word_index - 3), word_index):
        if words[i] in negation_words:
            return True
    return False

def score_sentences(text, global_freq, high_value_words, adjective_list):
    """Score sentences: normalized frequency + 3x high-value boost + 1.5x adjective bonus."""
    sentences = sent_tokenize(text)
    scores = {}
    has_negation = {}
    
    for sentence in sentences:
        words_in_sentence = word_tokenize(sentence.lower())
        raw_score = 0
        negation_found = False
        adjective_count = 0
        
        for idx, word in enumerate(words_in_sentence):
            base_freq = global_freq.get(word, 0)
            adjective_count += word in adjective_list
            
            if word in high_value_words:
                raw_score += base_freq * 3
                if detect_negation(sentence, idx):
                    negation_found = True
            else:
                raw_score += base_freq
        
        normalized_score = raw_score / len(words_in_sentence) if len(words_in_sentence) > 0 else 0
        if adjective_count > 0:
            normalized_score *= 1.5
        
        scores[sentence] = normalized_score
        has_negation[sentence] = negation_found
    
    return scores, has_negation

# Apply sentence scoring
df['sentence_scores'] = df['paragraph'].apply(
    lambda text: score_sentences(text, global_word_freq, high_value_sentiment, adjectives)[0]
)
df['has_negation'] = df['paragraph'].apply(
    lambda text: score_sentences(text, global_word_freq, high_value_sentiment, adjectives)[1]
)

## 4.6 Extract Top Sentence

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

def get_top_sentence(row):
    """Extract top sentence using: score, negation flag, sentiment intensity, order."""
    scores = row['sentence_scores']
    negations = row['has_negation']
    
    if not scores:
        return ""
    
    max_score = max(scores.values())
    tied_sentences = [s for s, score in scores.items() if score == max_score]
    
    if len(tied_sentences) == 1:
        return tied_sentences[0]
    
    sid = SentimentIntensityAnalyzer()
    best_sentence = max(tied_sentences, key=lambda s: (
        negations.get(s, False),
        abs(sid.polarity_scores(s)['compound']),
        -list(scores.keys()).index(s)
    ))
    return best_sentence

df['top_sentence'] = df.apply(get_top_sentence, axis=1)

,paragraph,top_sentence
0,I appreciated the eco-friendly initiatives the...,I appreciated the eco-friendly initiatives the...
1,"Our room was ready on time, but we weren't off...","Our room was ready on time, but we weren't off..."
2,We spent a rainy afternoon in the hotel librar...,We spent a rainy afternoon in the hotel librar...
3,Our room's balcony was the perfect spot for a ...,Our room's balcony was the perfect spot for a ...
4,The water pressure in the shower was a bit wea...,It's a standard hotel that doesn't take many r...


## 4.7 Sentiment Analysis of Top Sentence

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

nltk.download('vader_lexicon')

sid = SentimentIntensityAnalyzer()

# Get compound scores from all top sentences
compound_scores = [sid.polarity_scores(s)['compound'] for s in df['top_sentence']]

# Fine-tuned thresholds
POS_THRESHOLD = 0.05
NEG_THRESHOLD = -0.05

# Classify sentiments
def analyze_sentiment_vader(text):
    score = sid.polarity_scores(text)['compound']
    if score >= POS_THRESHOLD:
        return 'Positive'
    elif score <= NEG_THRESHOLD:
        return 'Negative'
    else:
        return 'Neutral'

df['predicted_sentiment'] = df['top_sentence'].apply(analyze_sentiment_vader)

print(f"Classification Thresholds:")
print(f"  Positive: ≥ {POS_THRESHOLD}")
print(f"  Negative: ≤ {NEG_THRESHOLD}")
print(f"  Neutral: {NEG_THRESHOLD} < score < {POS_THRESHOLD}")


VADER SENTIMENT ANALYSIS WITH REFINED THRESHOLDS

Thresholds Applied:
  Positive threshold: >= 0.05
  Negative threshold: <= -0.05
  Neutral window: -0.05 < score < 0.05
  Negative class F1-Score: 0.5714


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/kentbaluyot/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


# 5) Standardized Assignment Output

In [19]:
# Use the actual_sentiment column loaded from CSV (now correctly aligned!)
print("\n" + "=" * 80)
print("EVALUATION METRICS")
print("=" * 80)
print(f"\nAccuracy: {accuracy_score(df['actual_sentiment'], df['predicted_sentiment']):.4f}")
print("\nDetailed Classification Report:")
print(classification_report(df['actual_sentiment'], df['predicted_sentiment']))

# Confusion Matrix
from sklearn.metrics import confusion_matrix
print("\nConfusion Matrix (Rows: Positive, Neutral, Negative):")
cm = confusion_matrix(df['actual_sentiment'], df['predicted_sentiment'], 
                     labels=['Positive', 'Neutral', 'Negative'])
print(cm)
print("=" * 80)


EVALUATION METRICS

Accuracy: 0.5800

Detailed Classification Report:
              precision    recall  f1-score   support

    Negative       0.88      0.42      0.57        33
     Neutral       0.44      0.48      0.46        33
    Positive       0.58      0.82      0.68        34

    accuracy                           0.58       100
   macro avg       0.63      0.58      0.57       100
weighted avg       0.63      0.58      0.57       100


Confusion Matrix (Rows: Positive, Neutral, Negative):
[[28  6  0]
 [15 16  2]
 [ 5 14 14]]


In [21]:
# Format final output with required column names and ensure Email # is sequential
final_output_formatted = df[['email_num', 'paragraph', 'top_sentence', 'predicted_sentiment']].copy()

# Ensure Email # is sequential (1 to 100)
final_output_formatted['email_num'] = range(1, len(final_output_formatted) + 1)

final_output_formatted.rename(columns={
    'email_num': 'Email #',
    'paragraph': 'Paragraph',
    'top_sentence': 'Top Sentence',
    'predicted_sentiment': 'Sentiment'
}, inplace=True)

# Save to CSV
final_output_formatted.to_csv('assignment_output.csv', index=False)

# Final output - clean display with sequential Email # and summary statistics
print("\n" + "=" * 220)
print("FINAL ASSIGNMENT OUTPUT (Emails 1-100 with Extracted Top Sentences and Predicted Sentiments)")
print("=" * 220)
print(final_output_formatted.to_string(index=False))
print("=" * 220)

# Display summary statistics
print(f"\n✓ Successfully processed {len(final_output_formatted)} reviews")
print(f"✓ Email # range: 1 to {len(final_output_formatted)}")
print(f"\nSentiment Distribution (Predictions):")
sentiment_counts = final_output_formatted['Sentiment'].value_counts().sort_index()
for sentiment, count in sentiment_counts.items():
    percentage = (count / len(final_output_formatted)) * 100
    print(f"  {sentiment:8s}: {count:3d} reviews ({percentage:5.1f}%)")
print(f"\n✓ Output saved to 'assignment_output.csv'")



FINAL ASSIGNMENT OUTPUT (Emails 1-100 with Extracted Top Sentences and Predicted Sentiments)
 Email #                                                                                                                                                                                                                                                                                Paragraph                                                                                                                Top Sentence Sentiment
       1                        I appreciated the eco-friendly initiatives the hotel has implemented throughout the property. They use glass water bottles instead of plastic and have a very clear recycling program in the rooms. It's nice to stay at a place that prioritizes sustainability.                               I appreciated the eco-friendly initiatives the hotel has implemented throughout the property.  Positive
       2                                                  

# 6) Model Summary

## Pipeline Overview

This NLP solution implements a **3-stage extraction and classification framework**:

### **Stage 1: Preprocessing**
Clean and tokenize reviews into meaningful words:
- Sentence Tokenization → Word Tokenization → Stop Word Removal

### **Stage 2: Sentence Scoring**
Identify the most impactful sentence in each review:
- Calculate word frequencies across the corpus
- Score sentences: `(Sum of Word Frequencies) / (Total Words)` 
- Apply 3x weight to high-value sentiment words
- Apply 1.5x multiplier for sentences with high adjective density

### **Stage 3: Sentiment Classification**
Classify the extracted sentence using VADER:
- **Positive**: Compound score ≥ +0.05
- **Negative**: Compound score ≤ -0.05
- **Neutral**: -0.05 < Compound score < +0.05

See README for detailed methodology, performance analysis, and insights.